# Qwen 3.5 4B через llama.cpp

1. Запусти первую ячейку — она поднимет `llama-server`.
2. После сообщения `Сервер готов` запускай ячейку с чатом.
3. В конце можно остановить сервер последней ячейкой.

> Ноутбук предполагает, что он лежит в корне проекта рядом с папками `llama.cpp` и `models`.

In [ ]:
import subprocess
from urllib.request import urlopen
from urllib.error import URLError

SERVER_URL = "http://127.0.0.1:8080/health"

try:
    urlopen(SERVER_URL, timeout=1)
    print("llama-server уже запущен")
except URLError:
    server = subprocess.Popen([
        r".\llama.cpp\llama-server.exe",
        "--model", r".\models\Qwen_Qwen3.5-4B-Q4_K_M.gguf",
        "--alias", "qwen3.5-4b",
        "--host", "127.0.0.1",
        "--port", "8080",
        "--ctx-size", "4096",
        "--threads", "4",
        "--threads-batch", "4",
        "--reasoning", "off",
    ])

    print("llama-server запускается")

In [ ]:
from openai import OpenAI


client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
    timeout=300.0,
)

print("Отправляю запрос модели...")

response = client.chat.completions.create(
    model="qwen3.5-4b",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты преобразуешь неструктурированные данные "
                "в строго структурированный результат."
            ),
        },
        {
            "role": "user",
            "content": (
                "Преобразуй запись в JSON: "
                "дата 15.07.2026, артикул A-17, "
                "сумма 1 250,50 руб."
            ),
        },
    ],
    temperature=0.1,
    max_tokens=256,
)

text = response.choices[0].message.content

if text is None:
    raise RuntimeError("Модель вернула пустой ответ")

print("\nОтвет модели:\n")
print(text)

## Остановка сервера

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
server.terminate()